# Annotation inspection analysis

Load pair-level summary and per-cell UMAP parquet from `run_annotation_inspection_pipeline.py`, then explore cytescore vs confidence and plot lightweight UMAPs without reloading h5ad files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from shared.repo import REPO_ROOT

In [ ]:
RUN_DIR = REPO_ROOT / "output/annotation_inspection_pipeline" / "REPLACE_WITH_RUN_TIMESTAMP"

summary = pd.read_csv(RUN_DIR / "summary.csv")
extremes = pd.read_csv(RUN_DIR / "extremes.csv")
umap_cells = pd.read_parquet(RUN_DIR / "umap_cells.parquet")

summary.head()

## Pair-level groupbys

In [ ]:
summary.groupby(["cytetype_confidence", "accession"], observed=True)["cytescore_similarity"].mean().unstack().head()

In [ ]:
summary.groupby("cytetype_confidence", observed=True)["cytescore_similarity"].describe()

## Extremes table

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(extremes)

## Lightweight UMAP plots

In [ ]:
ACCESSION = umap_cells["accession"].iloc[0]
cells = umap_cells[umap_cells["accession"] == ACCESSION]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
plots = [
    ("cell_type", f"{ACCESSION}: STATE labels"),
    ("cytetype_annotation_leiden_merged", f"{ACCESSION}: CyteType annotations"),
    ("cytescore_similarity", f"{ACCESSION}: CyteOnto cytescore"),
    ("cytetype_confidence", f"{ACCESSION}: CyteType confidence"),
]

confidence_palette = {"Low": "#d73027", "Moderate": "#fee08b", "High": "#1a9850"}

for ax, (column, title) in zip(axes.ravel(), plots, strict=True):
    if column == "cytetype_confidence":
        for label, color in confidence_palette.items():
            sub = cells[cells[column] == label]
            ax.scatter(sub["umap_x"], sub["umap_y"], s=3, c=color, label=label, linewidths=0)
        ax.legend(markerscale=3, loc="best")
    elif column == "cytescore_similarity":
        sc = ax.scatter(
            cells["umap_x"],
            cells["umap_y"],
            c=cells[column],
            s=3,
            cmap="viridis",
            linewidths=0,
        )
        fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    else:
        for label in cells[column].astype(str).unique():
            sub = cells[cells[column].astype(str) == label]
            ax.scatter(sub["umap_x"], sub["umap_y"], s=3, label=label, linewidths=0)
        ax.legend(markerscale=3, loc="best", fontsize=6)
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")

fig.tight_layout()